In [1]:
from pathlib import Path
import torch
from ultralytics import YOLO

# Пути
PROJECT_DIR = Path("/home/Yolo_26")
MODEL_PATH = PROJECT_DIR / "runs_yolo26" / "trial_0005" / "weights" / "best.pt"
DATA_YAML = Path("/home/dataset/data.yaml")
OUTPUT_DIR = PROJECT_DIR / "test_best"

# Параметры
IMG_SIZE = 768
BATCH = 16
DEVICE = 0 if torch.cuda.is_available() else "cpu"

print("=" * 70)
print("ТЕСТ ЛУЧШЕЙ МОДЕЛИ YOLO26")
print("=" * 70)
print(f"Модель:   {MODEL_PATH}")
print(f"Датасет:  {DATA_YAML}")
print(f"GPU:      {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

if not MODEL_PATH.exists():
    raise FileNotFoundError(f"Не найдена модель:\n{MODEL_PATH}")

# Загружаем best.pt
model = YOLO(str(MODEL_PATH))

print("\nКлассы модели:")
for i, name in model.names.items():
    print(f"{i}: {name}")

# ============================================================
# ОЦЕНКА НА TEST
# ============================================================

print("\n" + "=" * 70)
print("ОЦЕНКА НА TEST")
print("=" * 70)

metrics = model.val(
    data=str(DATA_YAML),
    split="test",
    imgsz=IMG_SIZE,
    batch=BATCH,
    device=DEVICE,
    conf=0.001,
    iou=0.6,
    plots=True,
    save_json=True,
    project=str(OUTPUT_DIR),
    name="metrics",
    exist_ok=True,
    verbose=True,
)

# Основные метрики
print("\n" + "=" * 70)
print("ИТОГОВЫЕ МЕТРИКИ")
print("=" * 70)

print(f"Precision : {metrics.box.mp:.4f}")
print(f"Recall    : {metrics.box.mr:.4f}")
print(f"mAP50     : {metrics.box.map50:.4f}")
print(f"mAP50-95  : {metrics.box.map:.4f}")

# Метрики по классам
print("\n" + "=" * 70)
print("mAP50-95 ПО КЛАССАМ")
print("=" * 70)

for class_id, map_value in enumerate(metrics.box.maps):
    print(
        f"{class_id:2d} | "
        f"{model.names[class_id]:30s} | "
        f"{map_value:.4f}"
    )

# ============================================================
# ПРЕДСКАЗАНИЯ И ВИЗУАЛИЗАЦИЯ
# ============================================================

test_images = Path("/home/dataset/images/test")

images = [
    p for p in test_images.rglob("*")
    if p.is_file() and p.suffix.lower() in
    {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
]

print("\n" + "=" * 70)
print(f"ВИЗУАЛИЗАЦИЯ TEST: {len(images)} изображений")
print("=" * 70)

model.predict(
    source=[str(p) for p in images],
    imgsz=IMG_SIZE,
    batch=BATCH,
    device=DEVICE,
    conf=0.25,
    save=True,
    project=str(OUTPUT_DIR),
    name="predictions",
    exist_ok=True,
    verbose=True,
)

# Сохраняем краткий результат
summary = OUTPUT_DIR / "test_metrics.txt"

with open(summary, "w", encoding="utf-8") as f:
    f.write(f"Model: {MODEL_PATH}\n")
    f.write(f"Dataset: {DATA_YAML}\n")
    f.write(f"Test images: {len(images)}\n\n")
    f.write(f"Precision: {metrics.box.mp:.6f}\n")
    f.write(f"Recall: {metrics.box.mr:.6f}\n")
    f.write(f"mAP50: {metrics.box.map50:.6f}\n")
    f.write(f"mAP50-95: {metrics.box.map:.6f}\n\n")

    for class_id, map_value in enumerate(metrics.box.maps):
        f.write(
            f"{class_id}: {model.names[class_id]} — "
            f"mAP50-95 = {map_value:.6f}\n"
        )

print("\n" + "=" * 70)
print("ГОТОВО")
print("=" * 70)
print(f"Метрики и графики: {OUTPUT_DIR / 'metrics'}")
print(f"Визуализации:      {OUTPUT_DIR / 'predictions'}")
print(f"Результаты TXT:    {summary}")

ТЕСТ ЛУЧШЕЙ МОДЕЛИ YOLO26
Модель:   /home/Yolo_26/runs_yolo26/trial_0005/weights/best.pt
Датасет:  /home/dataset/data.yaml
GPU:      NVIDIA GeForce RTX 4090

Классы модели:
0: Экскаватор
1: Самосвал
2: Грузовой автомобиль
3: Кран
4: Фронтальный погрузчик
5: Автобетоносмеситель
6: Бульдозер
7: Прицеп
8: Каток
9: Автобетононасос

ОЦЕНКА НА TEST
Ultralytics 8.4.164 🚀 Python-3.12.13 torch-2.13.0+cu130 CUDA:0 (NVIDIA GeForce RTX 4090, 48536MiB)
YOLO26m summary (fused): 130 layers, 20,357,162 parameters, 0 gradients, 68.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 3583.6±2549.4 MB/s, size: 349.2 KB)
val: Scanning /home/dataset/labels/test... 2134 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 2134/2134 726.2it/s 2.9s0.1s
val: New cache created: /home/dataset/labels/test.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 134/134 8.8it/s 15.3s0.1s
                   all       2134       4701      0.912      0.839

[W929 00:15:33.110489865 CUDACachingAllocator.cpp:3933] memory allocation failed with OOM on device 0 while trying to allocate 80593551360 bytes (free: 13004046336, total: 50893815808).
[W929 00:15:33.381016728 CUDACachingAllocator.cpp:3933] memory allocation failed with OOM on device 0 while trying to allocate 80593551360 bytes (free: 28839641088, total: 50893815808).


OutOfMemoryError: CUDA out of memory. Tried to allocate 75.06 GiB. GPU 0 has a total capacity of 47.40 GiB of which 26.86 GiB is free. Process 4538 has 5.73 GiB memory in use. Including non-PyTorch memory, this process has 14.79 GiB memory in use. Of the allocated memory 14.29 GiB is allocated by PyTorch, and 20.53 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)